# Trabajo Práctico Integrador

## Port Log - Sprint 1

## Integrantes del grupo:

> Obtener el número de grupo de la pestaña del aula de "Trabajos Prácticos"
>
> Entrega UN SOLO integrante del grupo, debe ser siempre el mismo.
>
> Utilizar UN ÚNICO repositorio de github por grupo.

* Nombre y apellido
* Nombre y apellido
* ...


## Repositorio GITHUB

Especificar en la siguiente línea cual es la url para acceder al repositorio del grupo.



> COMPLETAR CON LA URL del repositorio







## Consideraciones Generales e Importantes

- Desarrollo base: Los puntos solicitados son la base del trabajo práctico. Es posible la ampliación sin restricciones del mismo.
- Para el traceo se deben agregar 1 [secrets](https://drlee.io/how-to-use-secrets-in-google-colab-for-api-key-protection-a-guide-for-openai-huggingface-and-c1ec9e1277e0) para la utilización de github.
  - GITHUB_TOKEN
- Organización del Notebook:
  - Agregar antes del ejercicio 01 una celda donde se importan todas librerías necesarias.
  - Utilizar un título para cada celda y escribir cada funcionalidad en una celda con título.
- La defensa es individual.
- Los grupos deben mantenerse a lo largo del trabajo integrador.
- La opción `DESACTIVAR_GIT_PUSH` solo será utilizada por los profesores.

## Versionado y Trabajo en Equipo


- Control de Versiones: Utilizar herramientas de versionado
- Colaboradores: Brindar acceso al repositorio a las siguientes cuentas:
  - lcd-had141@ugr.edu.ar
  - fpasinato@ugr.edu.ar
- La descripción del commit debe ser con el siguiente formato `"Día X: \<comentario del commit\>"`
- Changelog: Cada ejercicio/punto del trabajo práctico es considerado como un día distinto de trabajo. Debe quedar constancia en un archivo CHANGELOG.md de los cambios generados cada día. El changelog debe escribirse de tal forma que lo primero que se observe es el último cambio realizado, de esta forma apenas se abre el archivo se pueden leer los últimos cambios realizados. La forma de escritura debe ser un encabezado entre corchetes [Ejercicio X] y luego debajo los cambios realizado como bullets.
- El archivo README.md debe contener el sprint en el que nos encontramos y debe contener:
  - El objetivo.
  - La introducción y el contexto del sprint de trabajo.

## Criterios de evaluación

- Funcionalidad: el código debe ejecutar sin errores al dar "Ejecutar todo" (Ctrl+F9). Validar con Runtime -> Restart session and run all antes de entregar.
- Claridad: código organizado, nombres descriptivos, comentarios solo donde corresponde (funciones, clases, métodos). No se admite `git add .`.
- Estilo: [type hints](https://docs.python.org/3/library/typing.html), [pep8](https://peps.python.org/pep-0008/).

- Herramientas: librerías vistas en la cursada. Sin gdrive ni herramientas alternativas.

## Condiciones de Entrega
- Archivo `.ipynb`, ejecutado y con datos precargados, sobre copia de la plantilla original manteniendo la nomenclatura del nombre.
- Canal: aula virtual. Correo a lcd-had141@ugr.edu.ar solo si el plazo finalizó. No se aceptan enlaces.
- Los trabajos que no cumplan con los requisitos no serán tomados en cuenta como nota final. Solo serán corregidos y evaluados con nota para la devolución correspondiente. Se considera incumplimiento a:
  - Entregas fuera de término.
  - Trabajos que no se entreguen sobre la plantilla de trabajo estipulada.
  - Trabajos que no se ejecuten completos (que contengan errores intermedios al dar "Ejecutar todo").

---

## Extras

- [Manual Supabase](https://colab.research.google.com/drive/1bAL9yrkwByR3Qf6nviotKErYFcLAzcnL).
- [SQLAlchemy](https://colab.research.google.com/drive/1QBzLEKc6juktyqMYO6nz8QVMm1eMzvWY)
- Base de datos vectoriales chromadb [BaseDeDatosVectorialPPT](https://colab.research.google.com/drive/1nazuIlHGC_Od0Kbf5nPp6yEMk5dfJfQY) y [BaseDeDatosVectoriales](https://colab.research.google.com/drive/1z26X154sj9UAWhjpoUl9Ov2jnVxTm4iA)



## Objetivo

Aplicar conocimientos de bases de datos relacionales (SQLAlchemy), versionado de datos (DVC) y búsqueda por similitud visual (ChromaDB) para profesionalizar el sistema portuario.

## Introducción y Contexto del problema


### Sprint 3 — Desarrollar

El sistema ha crecido en volumen y complejidad. El equipo de desarrollo portuario necesita:

1. **Persistencia estructurada**: pasar de CSVs a una base de datos relacional con ORM.
2. **Auditoría de cambios**: poder rastrear qué versión del dataset se usó en cada análisis.
3. **Identificación visual de buques**: dado que la OCR no siempre funciona bien, el sistema debe poder identificar un buque por similitud visual de su imagen, sin depender del texto.

In [ ]:
#@title Instalación de paquetes
!pip install -q chromadb open-clip-torch dvc[s3]

In [ ]:
#@title Agregamos variables de entorno y secrets
import os
from google.colab import userdata

HOME=os.getenv('HOME', '/tmp')

HOOKS=f"{HOME}/.git-templates/hooks"
CMD_BIN=f"{HOME}/bin"

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
try:
  DESACTIVAR_GIT_PUSH = bool(userdata.get('DESACTIVAR_GIT_PUSH'))
except userdata.SecretNotFoundError:
  DESACTIVAR_GIT_PUSH = False
os.environ['DESACTIVAR_GIT_PUSH'] = str(DESACTIVAR_GIT_PUSH)
os.environ['HOOKS'] = HOOKS
os.environ['CMD_BIN'] = CMD_BIN
os.environ['PATH'] = f"{CMD_BIN}:" + os.environ['PATH']

In [ ]:
#@title Creación del hook para que durante la corrección no se realicen modificaciones al repositorio
%%bash
mkdir -p $HOOKS
cat << EOF > $HOOKS/pre-push
#!/bin/sh
if [ "$DESACTIVAR_GIT_PUSH" = "'True'" ] || [ "$DESACTIVAR_GIT_PUSH" = "True" ]; then
    echo "[GIT HOOK - WARNING] Pushes are disabled for this repository."
    exit 1
fi
EOF

chmod +x $HOOKS/pre-push

### DVC
mkdir -p $CMD_BIN
cat << 'EOF' > $CMD_BIN/dvc
#!/bin/sh
if [ "$1" = "push" ] && { [ "$DESACTIVAR_GIT_PUSH" = "True" ] || [ "$DESACTIVAR_GIT_PUSH" = "'True'" ]; }; then
    echo "\n=========================================================="
    echo "[BLOQUEO] 'dvc push' desactivado para esta corrección."
    echo "==========================================================\n"
    exit 0
fi
# Reemplaza aquí por la ruta real de DVC en Colab si no está en /usr/local/bin/dvc
/usr/local/bin/dvc "$@" 2>/dev/null || dvc "$@"
EOF

chmod +x $CMD_BIN/dvc

In [ ]:
!pip install -q dvc[s3]

## Ejercicio 01

> Puntos: 1

Crear la rama `Sprint_3` partiendo de `Sprint_2`.

Verificar el acceso a todos los datasets generados en sprints anteriores:
- `port_log/data/interim/port_movements.csv`
- `port_log/data/processed/port_movements_image.csv`
- `port_log/data/interim/group_images.json`
- `port_log/reports/summary_sprint1.csv`

Mostrar para cada dataset: cantidad de registros y fecha de última modificación del archivo.

In [ ]:
#@title Posicionarse en directorio de trabajo y agregar la siguiente configuración a git
if DESACTIVAR_GIT_PUSH:
  !git config --global core.hooksPath {HOOKS}

## Ejercicio 02

> Puntos: 1

Utilizar supabase como repositorio S3.

Migrar a DVC los siguientes archivos binarios (cada uno en una celda separada con su commit en CHANGELOG.md):
- Imágenes en `port_log/data/raw/imgs`.
- Gráficos en `port_log/data/interim/plots`.

Adicionalmente, versionar con DVC el archivo `port_log/data/processed/port_movements_image.csv` como dato procesado final, ya que es el output principal del pipeline. Justificar en un comentario por qué este archivo también debe versionarse con DVC y no solo con Git.

## Ejercicio 03

> Puntos: 1

Diseñar el **modelo lógico** de las siguientes entidades a partir de `port_log/data/processed/port_movements_image.csv`:

- **Buque**: embarcación registrada en el sistema.
- **Movimiento**: cada ingreso al muelle con su velocidad, tonelaje y estado.
- **Radar**: dispositivo de detección móvil, puede operar en distintos muelles.
- **Evidencia**: imagen asociada a un movimiento (opcional).

Relaciones a considerar:
- Un buque puede tener muchos movimientos.
- Un movimiento pertenece a un único radar.
- Un movimiento puede tener a lo sumo una evidencia.
- Un movimiento ocurre en un único muelle.
- Un radar puede registrar movimientos en distintos muelles.

Presentar el modelo como diagrama o descripción textual con atributos y cardinalidades. **No es necesario escribir código en este punto.**

## Ejercicio 04

> Puntos: 1

Implementar el **modelo relacional** con SQLAlchemy según el diseño del ejercicio anterior.

Cada modelo debe incluir:
- Clave primaria.
- Claves foráneas y relaciones (`relationship`).
- Método `__repr__` para mejor legibilidad.

Luego completar la función `procesar_fila_csv` que mapea una fila del CSV a instancias de los modelos.

In [ ]:
#@title Función procesar_fila_csv
def procesar_fila_csv(row: dict) -> Movimiento:
  """Mapea un diccionario de fila CSV a instancias del modelo.

  Args:
      row (dict): Fila del archivo port_movements_image.csv.

  Returns:
      Movimiento: Instancia con relaciones Buque, Radar y Evidencia.
  """
  pass

In [ ]:
#@title Validación
row_ejemplo = {
  'movimiento_id': 'MOV-00042',
  'matricula': 'MSC-GENOVA',
  'fecha_ingreso': '2023-08-14',
  'hora_ingreso': '03:15',
  'fecha_egreso': '2023-08-15',
  'duracion_horas': 20.5,
  'tonelaje_declarado': 4800.5,
  'velocidad_ingreso': 14.2,
  'velocidad_maxima_muelle': 12,
  'exceso_velocidad_real': 2.2,
  'exceso_velocidad': 1.6,
  'muelle': 'MUELLE-B',
  'tipo_carga': 'SOJA',
  'origen': 'ROTTERDAM',
  'radar_id': 'R-03',
  'estado_despacho': 'PENDIENTE',
  'imagen': 'foto_buque_042.jpg',
  'matricula_imagen': 'MSC-GEN0VA',
  'ratio': 0.90,
  'grupo_imagen': 'plate'
}

mov:Movimiento = procesar_fila_csv(row_ejemplo)
print(f"Movimiento: {mov.movimiento_id}")
print(f"Buque: {mov.buque.matricula}")
print(f"Radar: {mov.radar.radar_id} en {mov.muelle}")
print(f"Duración estadía: {mov.duracion_horas}hs")
if mov.evidencia:
  print(f"Evidencia: {mov.evidencia.archivo_imagen} "
        f"(ratio {mov.evidencia.ratio:.0%}, "
        f"grupo: {mov.evidencia.grupo})")

## Ejercicio 05

> Puntos: 1

Crear y poblar la base de datos `transito_portuario` con los datos de `port_log/data/processed/port_movements_image.csv`.

- Crear la base de datos con SQLAlchemy y generar las tablas automáticamente.
- Migrar los datos desde el CSV usando `procesar_fila_csv`.
- Validar la cantidad de registros insertados en cada tabla:
```
Buques insertados:     XX
Radares insertados:    XX
Movimientos insertados: XX
Evidencias insertadas: XX
```

## Ejercicio 06

> Puntos: 1

Implementar consultas SQLAlchemy sobre la base de datos:

- Top 10 buques con mayor cantidad de infracciones, ordenado de forma descendente:
```
--- Top 10 buques con más infracciones ---
Matrícula          | Cantidad  
------------------ | ----------
XXXXXXXXXX         | YY
```



- Top 10 movimientos más recientes sin evidencia, ordenado de forma descendente por fecha:
```
--- Total sin evidencia: XXX ---
Movimiento         | Fecha      | Matrícula
------------------ | ---------- | ----------
MOV-XXXXX          | YYYY-MM-DD | XXXXXXXXXX
```



- Radares más activos con la cantidad de muelles distintos en los que operaron, ordenados de forma descendente:
```
Radar      | Muelles distintos | Infracciones
---------- | ----------------- | ------------
R-XX       | X                 | ZZ
```



- Top 10 reincidentes en un período definido por el alumno, descendente:
```
--- Top 10 reincidentes (YYYY-MM-DD a YYYY-MM-DD) ---
Matrícula          | Cantidad  
------------------ | ----------
```



- Promedio de exceso de velocidad por muelle, descendente:
```
--- Exceso promedio por muelle ---
Muelle     | Exceso Prom. (nudos)
---------- | --------------------
MUELLE-X   | XX.XX
```



- Porcentaje de infracciones confirmadas visualmente:
```
--- Cobertura de evidencia ---
Total infracciones:       XXXX
Con imagen asociada:       XXX
Porcentaje confirmado:  XX.XX%
```

## Ejercicio 07

> Puntos: 1

Crear la base de datos vectorial `buque_vectorial` con ChromaDB usando el modelo OpenCLIP.

Para el almacenamiento:
- Usar el `id` del **Buque** (de la base relacional) como identificador del documento.
- Generar el embedding a partir de la imagen `complete` del buque (si tiene más de una, usar la de mayor ratio de coincidencia).
- Si un buque no tiene imagen `complete` asociada, no incluirlo en la base vectorial.

Al finalizar mostrar:
```
Buques en base relacional:    XX
Buques en base vectorial:     XX
Cobertura vectorial:       XX.XX%
```

## Ejercicio 08

> Puntos: 2

Implementar la función `buscar_buque_por_imagen` que:
1. Recibe la ruta a una imagen.
2. Genera su embedding con OpenCLIP.
3. Consulta la base vectorial para obtener el `buque_id` más similar.
4. Recupera todos los datos del buque desde la base relacional (incluyendo sus movimientos y evidencias).
5. Retorna un diccionario con la información completa.

In [ ]:
#@title Validación
result = buscar_buque_por_imagen(
  'port_log/data/raw/imgs/completes/0001_complete_patente.jpg'
)
print(result)

# También probar con una imagen plate (el sistema debe
# manejar el caso y explicar por qué puede fallar)
result2 = buscar_buque_por_imagen(
  'port_log/data/raw/imgs/plates/0001_plate_patente.jpg'
)
print(result2)

## Ejercicio 09

> Puntos: 1

Redactar una conclusión integradora del trabajo completo (3 sprints). Debe incluir:

- ¿Qué problema resuelve el sistema desarrollado de extremo a extremo?
- ¿Qué ventaja concreta aporta la base vectorial frente a solo usar OCR para identificar buques?
- ¿Cuáles son las principales limitaciones del sistema actual?
- Si tuvieras que poner este sistema en producción real en el Puerto de Rosario, ¿qué cambiarías y por qué?